## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
from haystack import Pipeline
from haystack.utils import Secret
from haystack.components.writers import DocumentWriter

In [ ]:
from sem_rag.config.settings import *

# Indexing

In [ ]:
from docling.chunking import HybridChunker
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

In [ ]:
QUESTION = "Which are the main AI models in Docling?"

In [ ]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(":memory:", embedding_dim=4096)

In [ ]:
converter = DoclingConverter(export_type=ExportType.DOC_CHUNKS)

indexing_embedder = VLLMDocumentEmbedder(model=VLLM.EMBEDDING_MODEL,
                                         api_key=Secret.from_token(VLLM.API_KEY),
                                         api_base_url=VLLM.API_URL)

writer = DocumentWriter(document_store=document_store)

In [ ]:
indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("embedder", indexing_embedder)
indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

In [ ]:
indexing_pipeline.show()

In [ ]:
indexing_pipeline.run(
    {
        "converter": {"sources": ["https://arxiv.org/pdf/2408.09869"]},
    })

# RAG

In [ ]:
from haystack.dataclasses import ChatMessage
from haystack.components.builders import AnswerBuilder
from haystack.components.builders.chat_prompt_builder import ChatPromptBuilder

from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [ ]:
chat_template = [
    ChatMessage.from_system(
        """
        You are a helpful AI assistant.
        Use the provided documents to answer the question.
        If the answer is not in the documents, say so.
        """
    ),
    ChatMessage.from_user(
        """
        Documents:
        {% for doc in documents %}
            {{ doc.content }}
        {% endfor %}

        Question:
        {{query}}
        """
    )
]

rag_pipeline = Pipeline()

In [ ]:
rag_embedder = VLLMTextEmbedder(model=VLLM.EMBEDDING_MODEL,
                                api_key=Secret.from_token(VLLM.API_KEY),
                                api_base_url=VLLM.API_URL)

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=5)

prompt_builder = ChatPromptBuilder(template=chat_template, required_variables={"query", "documents"})

llm = VLLMChatGenerator(model=VLLM.GEMMA_MODEL,
                        api_key=Secret.from_token(VLLM.API_KEY),
                        api_base_url=VLLM.API_URL)

answer_builder = AnswerBuilder()

In [ ]:
rag_pipeline.add_component("embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

In [ ]:
rag_pipeline.show()

In [ ]:
response = rag_pipeline.run(
    {
        "embedder": {"text": QUESTION},
        "prompt_builder": {"query": QUESTION},
        "answer_builder": {"query": QUESTION},
    })

In [ ]:
response["answer_builder"]["answers"][0].data

In [ ]:
docs = document_store.filter_documents()

print(len(docs))

print(docs[0].embedding is not None)

print(type(docs[0].embedding))

print(len(docs[0].embedding) if docs[0].embedding else None)

print(docs[0].embedding[:10] if docs[0].embedding else None)